# The Midas Touch: Hype vs Reality 👑✨

**Data Storytelling by Andres Felipe Rosero Lopez**

---

### 🎯 The Conflict: Hype (Expectation) vs Reality (Score)

Every great story needs a conflict. In the world of anime, this conflict is often between **Hype** (how many people follow a work) and **Reality** (the actual perceived quality, the score).

In this notebook, we will explore:
1.  **Hype vs Reality**: Does a big name attract people, but only quality keeps them?
2.  **The Midas Touch**: Are great directors (Miyazaki, Shinkai, Yuasa) truly infallible?
3.  **One Hit Wonders**: The curse of the masterpiece. What happens if we remove the main work from authors?
4.  **The Sounds of Success**: The impact of composers (e.g., Hiroyuki Sawano) on the score.
5.  **Star Value (VORD)**: A "sabermetric" comparison between directors and studios.
6.  **Studio DNA**: Does every studio have a quality fingerprint?
7.  **Originals vs Adaptations**: Is an original anime a risk worth taking?

**Goal**: To demonstrate that *"Fame opens the door, but consistency builds the temple."*

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import warnings

# Add parent directory to path to import src
sys.path.append('..')
from src.data import load_processed

# Settings
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Custom Color Palette
hype_color = '#FFD700' # Gold for Midas
reality_color = '#2E8B57' # Green for Reality

print("Environment configured. ✅")

Environment configured. ✅


## 1. Data Loading & Cleaning 🧹

We load the main datasets and those related to staff/people.
The main challenge here is **name normalization**, since we often find different formats (Family Name Given Name vs Given Name Family Name).

In [17]:
print("Loading core datasets...")
try:
    df_details = load_processed("details")
    df_stats = load_processed("stats")
    
    # Merge Core
    df = df_details.merge(df_stats, on="anime_id", how="inner")
    print(f"Main dataset loaded: {df.shape[0]} anime.")
    
except Exception as e:
    print(f"Error loading core data: {e}")

print("Loading people & staff datasets...")
try:
    # Loading Staff/Person data
    # We need to link anime -> staff -> person details
    df_person_details = load_processed("person_details")
    df_person_works = load_processed("person_anime_works")
    
    print(f"Person Details: {df_person_details.shape[0]} people.")
    print(f"Person Works: {df_person_works.shape[0]} records.")
    
except Exception as e:
    print(f"Error loading people data: {e}")

Loading core datasets...
Main dataset loaded: 28939 anime.
Loading people & staff datasets...
Person Details: 76696 people.
Person Works: 458091 records.


In [18]:
# Normalize Names Function
def normalize_name(name):
    """
    Normalizes names to 'Given Name Family Name' format.
    Handles 'Family, Given' format by swapping.
    """
    if pd.isna(name):
        return "Unknown"
    
    name = str(name).strip()
    
    if ',' in name:
        parts = name.split(',')
        if len(parts) >= 2:
            # Assuming "Family, Given" -> return "Given Family"
            return f"{parts[1].strip()} {parts[0].strip()}"
            
    return name

# Apply Normalization
if 'name' in df_person_details.columns:
    df_person_details['normalized_name'] = df_person_details['name'].apply(normalize_name)
    print("Name normalization applied.")
else:
    print("Column 'name' not found in person_details. Please check columns:", df_person_details.columns)

Name normalization applied.


## Chapter 2: The Hype Factor 💥

**The Hypothesis**: "A big name attracts people (Members), but only quality keeps them (Score)."

We analyze the **Members/Score** ratio.
- A high number of members with a mediocre score indicates **Disappointed Hype**.
- We look for titles where expectations were sky-high but the result was... questionable.

In [19]:
# Filter for relevant anime (exclude extremely niche ones to avoid noise)
df_analysis = df[df['members'] > 10000].copy()

# Calculate Hype Ratio (Members per Score point)
df_analysis['hype_ratio'] = df_analysis['members'] / df_analysis['score']

# Identify High Hype, Low Score (The Flops)
top_hype_flops = df_analysis.sort_values(by='hype_ratio', ascending=False).head(20)

fig = px.scatter(df_analysis, x='score', y='members', 
                 hover_name='title', 
                 title='The Hype Landscape: Members vs Score',
                 color='score',
                 color_continuous_scale='RdYlGn',
                 log_y=True)

# Highlight the "Flops"
fig.add_trace(go.Scatter(x=top_hype_flops['score'], y=top_hype_flops['members'],
                         mode='markers',
                         marker=dict(color='red', size=10, symbol='x'),
                         name='High Hype / Low Score'))

fig.show()

## Chapter 3: The Seal of Quality (Ridgeline Plots) 🏔️

We analyze the consistency of great directors.
We use a **Ridgeline Plot** to visualize the distribution of scores for selected directors (e.g., Miyazaki, Shinkai, Yuasa, Araki).

- **Narrow and high mountain on the right**: Consistent and high-quality author.
- **Wide mountain**: Unpredictable author.

In [20]:
# Prepare Data for Directors
# We need to merge anime -> person_works (role='Director') -> person_details

# Filter works for Director role
director_works = df_person_works[df_person_works['position'].str.contains('Director', case=False, na=False)]

# Merge with Person Names
director_works = director_works.merge(df_person_details[['person_id', 'normalized_name']], 
                                      left_on='person_id', right_on='person_id', how='left')

# Merge with Anime Scores
director_scores = director_works.merge(df[['anime_id', 'title', 'score', 'studios']], on='anime_id', how='inner')

# Select Top Directors by number of works (min 5 works) to have meaningful distributions
director_counts = director_scores['normalized_name'].value_counts()
top_directors_list = director_counts[director_counts >= 5].index

# Specific Directors of Interest (User suggested + famous ones)
target_directors = ['Hayao Miyazaki', 'Makoto Shinkai', 'Masaaki Yuasa', 'Hideaki Anno', 'Mamoru Hosoda', 'Naoko Yamada', 'Shinichirou Watanabe']
# Check if they exist in our normalized names
available_directors = [d for d in target_directors if d in director_scores['normalized_name'].unique()]

plot_data = director_scores[director_scores['normalized_name'].isin(available_directors)]

if len(plot_data) > 0:
    fig = go.Figure()

    for director in available_directors:
        director_data = plot_data[plot_data['normalized_name'] == director]['score']
        # Use Violin plot as a proxy for Ridgeline in standard plotly if joyplot absent
        fig.add_trace(go.Violin(x=director_data, y0=0, 
                                name=director, 
                                side='positive',
                                orientation='h',
                                width=1.5,
                                points='all'))

    fig.update_layout(title='Director Consistency: Score Distributions',
                      xaxis_title='Score',
                      yaxis_title='Director',
                      showlegend=False)
    fig.show()
else:
    print("No data found for selected directors. Check name normalization.")

## Chapter 4: The Curse of the Masterpiece (One Hit Wonder) 🥀

If we remove the main work (the masterpiece), what remains of the author's prestige?
We calculate the "Prestige Drop".

In [21]:
# Re-use director_scores

prestige_analysis = []

for director in top_directors_list:
    d_data = director_scores[director_scores['normalized_name'] == director]
    if d_data.empty: continue
        
    avg_score_all = d_data['score'].mean()
    max_score = d_data['score'].max()
    
    # Remove the masterpiece (highest score)
    # Handle ties by removing only one instance or all? Let's remove the max value instances.
    d_data_no_masterpiece = d_data[d_data['score'] < max_score]
    
    if d_data_no_masterpiece.empty:
        avg_score_no_masterpiece = 0 # Or NaN
    else:
        avg_score_no_masterpiece = d_data_no_masterpiece['score'].mean()
        
    prestige_drop = avg_score_all - avg_score_no_masterpiece
    
    prestige_analysis.append({
        'Director': director,
        'Works Count': len(d_data),
        'Avg Score (All)': avg_score_all,
        'Avg Score (w/o Masterpiece)': avg_score_no_masterpiece,
        'Prestige Drop': prestige_drop,
        'Masterpiece Score': max_score
    })

df_prestige = pd.DataFrame(prestige_analysis)

# Filter directors with significant drops (Positive Prestige Drop means average goes DOWN without masterpiece)
# We look for the largest drops
one_hit_wonders = df_prestige[df_prestige['Works Count'] >= 3].sort_values(by='Prestige Drop', ascending=False).head(15)

fig = px.bar(one_hit_wonders, x='Director', y='Prestige Drop',
             title='The Curse of the Masterpiece: Drop in Average Score when removing the best work',
             color='Prestige Drop', color_continuous_scale='OrRd',
             hover_data=['Avg Score (All)', 'Avg Score (w/o Masterpiece)', 'Masterpiece Score'])
fig.show()

## Extension: Sounds of Success (Composers) 🎵

Does Hiroyuki Sawano (and other top composers) really raise the average?
Let's compare the score of anime with these composers vs the global average.

In [22]:
# Filter works for Music/Composer role
# Roles often include "Music", "Theme Song Performance", etc.
composer_works = df_person_works[df_person_works['position'].str.contains('Music', case=False, na=False)]

# Merge with Person Names
composer_works = composer_works.merge(df_person_details[['person_id', 'normalized_name']], 
                                      left_on='person_id', right_on='person_id', how='left')

# Merge with Anime Scores
composer_scores = composer_works.merge(df[['anime_id', 'title', 'score']], on='anime_id', how='inner')

target_composers = ['Hiroyuki Sawano', 'Yuki Kajiura', 'Yoko Kanno', 'Joe Hisaishi', 'Kensuke Ushio']

composer_comparison = []
global_avg = df['score'].mean()

for composer in target_composers:
    c_data = composer_scores[composer_scores['normalized_name'] == composer]
    if c_data.empty: continue
    
    avg_score = c_data['score'].mean()
    uplift = avg_score - global_avg
    
    composer_comparison.append({
        'Composer': composer,
        'Works Count': len(c_data),
        'Avg Score': avg_score,
        'Global Avg': global_avg,
        'Score Uplift': uplift
    })

df_composers = pd.DataFrame(composer_comparison)

fig = go.Figure()
fig.add_trace(go.Bar(x=df_composers['Composer'], y=df_composers['Score Uplift'], 
                     marker_color='teal', name='Score Uplift vs Average'))

fig.update_layout(title='The Composer Effect: Score Uplift vs Global Average',
                  yaxis_title='Score Points above Average')
fig.show()

## Special Chapter: Studio DNA 🧬

Do animation studios have a quality "fingerprint"? Let's analyze the consistency of the Big Names.
Is KyoAni really synonymous with constant excellence?

In [23]:
# Filter Top Studios
# Studio column often contains lists "Madhouse, MAPPA". We should explode them for accuracy,
# but for main studios usually they are listed first or alone.
# Let's check consistency of major studios.

target_studios = ['Kyoto Animation', 'Madhouse', 'MAPPA', 'Bones', 'Sunrise', 'A-1 Pictures', 'J.C.Staff', 'Studio Ghibli', 'ufotable', 'Production I.G']

# Create a dataset where we track scores for each of these studios
studio_scores = []

for studio in target_studios:
    # Simple contains match
    s_data = df[df['studios'].astype(str).str.contains(studio, na=False)]
    if len(s_data) > 0:
        temp_df = s_data[['title', 'score']].copy()
        temp_df['Studio'] = studio
        studio_scores.append(temp_df)

df_studio_dna = pd.concat(studio_scores)

# Order by median score for better visual
studio_order = df_studio_dna.groupby('Studio')['score'].median().sort_values(ascending=False).index

# Violin Plot for Studio DNA
fig = go.Figure()

for studio in studio_order:
    data = df_studio_dna[df_studio_dna['Studio'] == studio]['score']
    fig.add_trace(go.Violin(x=data, y0=0, name=studio, orientation='h', side='positive', width=1.3, points=False))

fig.update_layout(title='Studio DNA: Score Consistency & Quality',
                  xaxis_title='Score', 
                  yaxis_title='Studio',
                  showlegend=False,
                  height=800)
fig.show()

## Extra Chapter: VORD (Value Over Replacement Director) ⚾

Inspired by Sabermetrics ("Moneyball"): how much value does a director add compared to the "average" of their studio?
If Hayao Miyazaki directs a Ghibli film, how much score does it gain compared to a Ghibli film *not* directed by him?

**Formula**: `VORD = (Avg Score Director at Studio X) - (Avg Score Studio X without Director)`

In [24]:
# Calculate VORD for selected Directors
# Logic:
# 1. Iterate through directors.
# 2. Find their main studio (most frequent).
# 3. Calculate Avg Score of Director @ Studio.
# 4. Calculate Avg Score of Studio (excluding Director).
# 5. Difference is VORD.

vord_data = []

for director in target_directors:
    # Get director works
    d_works = director_scores[director_scores['normalized_name'] == director]
    if d_works.empty: continue
    
    # Get main studio (most frequent)
    # Studios col is comma separated string. Let's simplistically take the first one listed for grouping or explode.
    # For better accuracy, we look at the studio string.
    main_studio = d_works['studios'].mode()[0]
    if pd.isna(main_studio): continue
        
    # Clean studio name (first element if multiple)
    main_studio_clean = main_studio.split(',')[0].strip()
    
    # Get all works from that studio
    all_studio_works = df[df['studios'].str.contains(main_studio_clean, na=False)]
    
    # Exclude director's works
    studio_works_no_director = all_studio_works[~all_studio_works['anime_id'].isin(d_works['anime_id'])]
    
    if studio_works_no_director.empty: 
        # The director IS the studio (e.g. maybe Shinkai early Comix Wave?)
        continue
        
    score_director = d_works[d_works['studios'].str.contains(main_studio_clean, na=False)]['score'].mean()
    score_replacement = studio_works_no_director['score'].mean()
    
    vord = score_director - score_replacement
    
    vord_data.append({
        'Director': director,
        'Main Studio': main_studio_clean,
        'Director Avg': score_director,
        'Studio Baseline': score_replacement,
        'VORD': vord
    })

df_vord = pd.DataFrame(vord_data).sort_values(by='VORD', ascending=False)

fig = px.bar(df_vord, x='Director', y='VORD',
             title='VORD: Value Over Replacement Director (Score Impact vs Studio Average)',
             hover_data=['Main Studio', 'Director Avg', 'Studio Baseline'],
             color='VORD', color_continuous_scale='Viridis')
fig.show()

## Risk Chapter: Original vs Adaptation 🎲

Is creating an original anime a bigger gamble than adapting a popular manga?
We compare the **Standard Deviation** and the **Distribution** of scores by Source.

In [25]:
# Setup Source Group Mapping if not present (simulating run)
def map_source_group(source):
    if pd.isna(source): return "Unknown"
    source = str(source).strip()
    if source in ['Manga', '4-koma manga', 'Web manga', 'Digital manga']: return 'Manga First'
    elif source in ['Light novel', 'Novel', 'Visual novel', 'Book']: return 'Novel First'
    elif source == 'Original': return 'Original'
    elif source in ['Game', 'Card game']: return 'Game'
    else: return 'Other'

df['source_group'] = df['source'].apply(map_source_group)

# Filter main groups
main_sources = ['Manga First', 'Novel First', 'Original', 'Game']
df_risk = df[df['source_group'].isin(main_sources)]

# Box Plot to show variance (Risk)
fig = px.box(df_risk, x='source_group', y='score', 
             color='source_group',
             title='The Risk Profile: Score Distribution by Source Material',
             points='outliers') # outliers only to reduce rendering load

fig.update_layout(xaxis_title='Source Material',
                  yaxis_title='Score')
fig.show()

# Calculate Stats
risk_stats = df_risk.groupby('source_group')['score'].agg(['mean', 'std', 'count']).sort_values(by='std', ascending=False)
print("Risk Analysis Stats (Standard Deviation = Volatility):")
print(risk_stats)

Risk Analysis Stats (Standard Deviation = Volatility):
                  mean       std  count
source_group                           
Original      6.066450  0.878572   6288
Manga First   6.797031  0.837418   5740
Novel First   6.626904  0.792264   2988
Game          6.331161  0.710042   1291


## Conclusions: The Equation of Success

We have seen that:
1.  Hype (members) does not guarantee Quality (score), and flops from big names do exist.
2.  The "Midas" (Miyazaki, etc.) have a massive positive VORD: their name is worth more than the studio.
3.  Originals are the biggest gamble (high variance), while Manga offer a safer base but perhaps fewer peaks of absolute excellence (average vs outlier).

"Fame opens the door, but consistency builds the temple."